In [ ]:
import kagglehub
import numpy as np
import pandas as pd
import os

import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
# Task 1: Write your code here:

# Load the CSV file

path2 = os.path.join(path, 'Q1_data.csv')
df = pd.read_csv(path2)

In [ ]:
# Task 2: Write your code here:

print(f"Shape: {df.shape}")
df.head()

In [ ]:
# Task 3: Write your code here:

df.info()

In [ ]:
# Task 4: Write your code here:

df.describe()

In [ ]:
# Task 5: Write your code here:

feature_to_plot = "Delivery_Time"

print(f"delivery_time: {df['Delivery_Time'].sum()}")
print(f"Normal: {(df['Delivery_Time'] == 0).sum()}")

# Attack distribution
plt.figure(figsize=(8, 4))
plt.hist(df['Delivery_Time'].dropna(), bins=30, edgecolor='black')
plt.title(f"Distribution of {feature_to_plot}")
plt.xlabel('Delivery_Time')
plt.ylabel('Frequency')
plt.show()

In [ ]:
# Task 1: Write your code here:

# Define stat columns
stat_cols = ['Delivery_Time']

df_clean = df.dropna(subset=stat_cols).copy()
print(f"Shape after cleaning: {df_clean.shape}")


In [ ]:
# Task 2: Write your code here:
# Missing values
print("Missing values:")
print(df.isnull().sum())

#  Do we have missing values?
def check_missing_values(df):
  missing_values = df.isnull().sum()
  print("Missing Values per Column:")
  print(missing_values[missing_values > 0]) #return zero if there is not a missing value
  if missing_values.any(): #if there is a missing value enter here
    print("\nHandle Missing Values as needed.")
  else:
    print("\nNo Missing Values Found.")

check_missing_values(df)


In [ ]:
# Task 3: Write your code here:

# Do we have duplicate samples?
def check_duplicates(df):
  duplicates = df.duplicated().sum()
  print(f"Number of Duplicate Samples: {duplicates}")
  if duplicates > 0:
    print("Dropping Duplicates...")
    df.drop_duplicates(inplace=True)
    print("Duplicates Dropped.")
  else:
    print("No Duplicate Samples Found.")

check_duplicates(df)


In [ ]:
# Task 4: Write your code here:
# Encode categorical variables if needed (Bonus if used One Hot Encoding)

from sklearn.preprocessing import LabelEncoder

for col in ['Order_ID','Distance_km', 'Weather','Traffic_Level','Time_of_Day','Vehicle_Type','Preparation_Time_min','Courier_Experience_yrs','Delivery_Time']:
    df_clean[col] = df_clean[col].fillna('unknown')


print("Missing values remaining:", df_clean.isnull().sum().sum())
# Encode categorical columns - converts text to integers
categorical_cols = ['Order_ID','Distance_km', 'Weather','Traffic_Level','Time_of_Day','Vehicle_Type','Preparation_Time_min','Courier_Experience_yrs','Delivery_Time']
for col in categorical_cols:
    le = LabelEncoder()
    df_clean[col] = le.fit_transform(df_clean[col].astype(str))

df_clean.head()



In [ ]:
# Task 5: Write your code here:
# Apply feature scaling for all features (Use StandardScaler)
from sklearn.preprocessing import StandardScaler #import StandardScaler
data_for_scale = pd.DataFrame({"Feature_1": [11, 40, 19,12],"Feature_2": [2384, 439, 3282,576]})

print("Before scaling:")
data_for_scale

print('data before scaling:\n', data_for_scale) #show before scaling
standard_scaler = StandardScaler() # Instantiate StandardScaler
data_standard_scaled = standard_scaler.fit_transform(data_for_scale) # Apply fit_transform

print('\nData after scaling:\n', data_standard_scaled) #show after scaling


In [ ]:
# Task 6: Write your code here:
#Check for target imbalance and state if it is imbalanced or not (keep this cell empty if not needed)

# 1. Is the target imbalanced?
def check_target_imbalance(df, target_column):
  print("Target Distribution:")

  df[target_column].hist()  # Yeah you can just do this :)
  plt.show()

check_target_imbalance(df, "Air Quality")

In [ ]:
# Task 1: Write your code here:
import numpy as np # for random data generation

np.random.seed(42) # for reproducbility

X_reg = pd.DataFrame({"Feature_1": np.random.randn(500)}) # features
y_reg = 3 * X_reg["Feature_1"] + np.random.randn(500) * 0.5  # continous labels with noise


X_clf = pd.DataFrame({"Feature_1": np.random.randn(500)}) # features
y_clf = (X_clf["Feature_1"] > 0).astype(int) # labels as class 0 or 1


In [ ]:
# Task 2,3,4,5: Write your code here:

#2------------------------------------------------------------------
from sklearn.model_selection import KFold

# Use previously generated random data (example: regression data)
X, y = X_reg.copy(), y_reg.copy()

# Define K-Fold Cross Validation
kf = KFold(n_splits=5, shuffle=True, random_state=42)

# Iterate through folds
for fold, (train_idx, test_idx) in enumerate(kf.split(X), start=1):
    # indexing for each fold
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
    # print shapes
    print(f"Fold {fold}")
    print("  X_train shape:", X_train.shape)
    print("  X_test shape :", X_test.shape)
    print("  y_train shape:", y_train.shape)
    print("  y_test shape :", y_test.shape)
    print("-" * 30)

#3--------------------------------------------------------------------
# Import tree-based model for classification (Decision Tree Classifier)
from sklearn.tree import DecisionTreeClassifier

# Use previously generated random classification data
X, y = X_clf.copy(), y_clf.copy()

# Split ratio (80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(X, y,test_size=0.2,random_state=42,shuffle=True)

# Train Decision Tree Classifier
model = DecisionTreeClassifier(random_state=42) # instantiate
model.fit(X_train, y_train)                     # fit
y_pred = model.predict(X_test)                  # predict

# lets use confusion matrix to visualize the predictions
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

# Create confusion matrix
cm = confusion_matrix(y_test, y_pred, labels=model.classes_)

# use ConfusionMatrixDisplay for visualization
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=model.classes_)

disp.plot()
plt.show()

#4-----------------------------------------------------------------------------------
import torch
import torch.nn as nn
from sklearn.metrics import mean_absolute_error

# TODO: Set up the device (use GPU if available, otherwise CPU)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Get predictions on test set
model.eval()
with torch.no_grad():
  # TODO: Get predictions for X_test
  # [HINT]: Move X_test to device, pass through model, then move back to CPU
  predictions = model(X_test.to(device)).cpu().numpy()

# TODO: Calculate evaluation metrics
# [HINT]: Use sklearn metrics - mean_absolute_error, mean_squared_error, r2_score
mae = mean_absolute_error(y_test.numpy(), predictions)

print("Model Evaluation")
print(f"  MAE:  ${mae:.2f}")




In [ ]:
# Task 1: Write your code here:

In [ ]:
# Task 2: Write your code here:

In [ ]:
# Task Bonus: Write your code here: